In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Automated Resampling Calibration via 5-Fold Stratified CV (`models/calibrate_resampling_ratios_5fold_cv.ipynb`)

This notebook automatically tunes and calibrates Layer 1 **Class Weight Multiplier** and Layer 2 **Individual Resampling Ratios** using **5-Fold Stratified Cross-Validation** combined with **Optuna Bayesian Optimization**:

### Layer 1 Class Weighting Strategy (No Downsampling)
- **Full Training Data Intact**: Layer 1 is trained on complete data without downsampling.
- **Baseline Inverse Class Weight**: $\text{baseline\_weight} = \frac{N_{\text{non-ESI1}}}{N_{\text{ESI1}}}$.
- **Tunable Multiplier (`l1_weight_multiplier`)**: Iterates from `0.00` to `1.00` in `0.01` increments.
- **Applied `scale_pos_weight`**: $1.0 + (\text{baseline\_weight} - 1.0) \times \text{l1\_weight\_multiplier}$.

### Optimization Objective
- **Target Metric**: Maximize **5-Fold Out-Of-Fold (OOF) Macro Balanced Accuracy** of the **2-Tier Combined Soft Probability Model**:
  - $P(\text{ESI 1}) = P_{L1}(\text{ESI 1})$
  - $P(\text{ESI } c) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI } c), \quad c \in \{2, 3, 4, 5\}$
  - $\hat{y} = \arg\max(P_1, P_2, P_3, P_4, P_5)$

### Calibrated Hyperparameter Search Space
1. **`l1_weight_multiplier`**: $[0.00, 1.00]$ (Step `0.01`).
2. **`r_esi2`**: $[0.1, 1.0]$ (Step `0.1`).
3. **`r_esi3`**: $[0.1, 1.0]$ (Step `0.1`).
4. **`r_esi4`**: $[0.1, 1.0]$ (Step `0.1`).
5. **`r_esi5`**: $[0.1, 1.0]$ (Step `0.1`).

The winning optimal parameters are automatically written into `config/triage_conf.json`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Prepare 35 Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = pulse_last, resp_last = resp_last, spo2_last = spo2_last, sbp_last = sbp_last,
  pulse_min = pulse_min, resp_min = resp_min, spo2_min = spo2_min, sbp_min = sbp_min,
  pulse_max = pulse_max, resp_max = resp_max, spo2_max = spo2_max, sbp_max = sbp_max,
  hr_mean_to_last = t_hr - pulse_last, sbp_mean_to_last = t_sbp - sbp_last, spo2_mean_to_last = t_o2 - spo2_last, rr_mean_to_last = t_rr - resp_last,
  hr_range = pulse_max - pulse_min, rr_range = resp_max - resp_min, spo2_range = spo2_max - spo2_min, sbp_range = sbp_max - sbp_min,
  hr_last_to_min = pulse_last - pulse_min, rr_last_to_min = resp_last - resp_min, spo2_last_to_min = spo2_last - spo2_min, sbp_last_to_min = sbp_last - sbp_min,
  hr_last_to_max = pulse_last - pulse_max, rr_last_to_max = resp_last - resp_max, spo2_last_to_max = spo2_last - spo2_max, sbp_last_to_max = sbp_last - sbp_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
train_val_py <<- train_val_df
test_py      <<- test_df
cat(sprintf("Prepared Calibration Data: Development Split=%d rows, Holdout Test=%d rows\n", nrow(train_val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Optuna 5-Fold Stratified CV Calibration Loop in Python
# ---------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Retrieve data from R
try:
    pandas2ri.activate()
    train_val_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_val_py']))
except Exception:
    train_val_df = pd.DataFrame(r['train_val_py'])
feature_cols = [c for c in train_val_df.columns if c != 'target_col']
binary_cols  = ['gender', 'cc_breathingdifficulty']
cont_cols    = [c for c in feature_cols if c not in binary_cols]
X_raw = train_val_df[feature_cols].values
y_raw = train_val_df['target_col'].astype(str).values
y_int = train_val_df['target_col'].astype(int).values
# Pre-calculate 5-Fold Stratified Splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(skf.split(X_raw, y_raw))
def compute_macro_balanced_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    bal_accs = []
    for i in range(5):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((rec + spec) / 2.0)
    return np.mean(bal_accs)
def objective(trial):
    # Suggest Layer 1 Class Weight Multiplier (0.00 to 1.00 in 0.01 increments)
    l1_weight_multiplier = trial.suggest_float("l1_weight_multiplier", 0.00, 1.00, step=0.01)
    
    # Suggest Layer 2 Resampling Ratios (0.10 to 1.00 in 0.10 increments)
    r_esi2 = trial.suggest_float("r_esi2", 0.1, 1.0, step=0.1)
    r_esi3 = trial.suggest_float("r_esi3", 0.1, 1.0, step=0.1)
    r_esi4 = trial.suggest_float("r_esi4", 0.1, 1.0, step=0.1)
    r_esi5 = trial.suggest_float("r_esi5", 0.1, 1.0, step=0.1)
    
    oof_probs = np.zeros((len(train_val_df), 5))
    oof_preds = np.zeros(len(train_val_df), dtype=int)
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv_splits):
        df_tr = train_val_df.iloc[train_idx]
        df_vl = train_val_df.iloc[val_idx]
        
        scaler = StandardScaler()
        X_tr_cont = scaler.fit_transform(df_tr[cont_cols])
        X_vl_cont = scaler.transform(df_vl[cont_cols])
        
        X_tr = np.hstack([X_tr_cont, df_tr[binary_cols].values])
        X_vl = np.hstack([X_vl_cont, df_vl[binary_cols].values])
        
        y_tr = df_tr['target_col'].astype(str).values
        y_vl = df_vl['target_col'].astype(str).values
        
        # 1. Layer 1 LightGBM (ESI 1 Detector - Class Weighted WITHOUT Downsampling)
        y_l1_tr = (y_tr == '1').astype(int)
        n_pos = np.sum(y_l1_tr == 1)
        n_neg = np.sum(y_l1_tr == 0)
        baseline_weight = n_neg / max(n_pos, 1)
        actual_scale_pos_weight = 1.0 + (baseline_weight - 1.0) * l1_weight_multiplier
        
        lgb_l1 = lgb.LGBMClassifier(
            n_estimators=100,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=6,
            scale_pos_weight=actual_scale_pos_weight,
            random_state=42,
            verbose=-1
        )
        lgb_l1.fit(X_tr, y_l1_tr)
        
        # 2. Layer 2 Direct 4-Class LightGBM (Individual Resampled Fold)
        no_esi1_idx = (y_tr != '1')
        X_tr_l2_raw = X_tr[no_esi1_idx]
        y_l2_raw = y_tr[no_esi1_idx]
        
        idx_2 = np.where(y_l2_raw == '2')[0]
        idx_3 = np.where(y_l2_raw == '3')[0]
        idx_4 = np.where(y_l2_raw == '4')[0]
        idx_5 = np.where(y_l2_raw == '5')[0]
        
        max_class_cnt = max(len(idx_2), len(idx_3), len(idx_4), len(idx_5))
        target_cnts = [
            int(round(max_class_cnt * r_esi2)),
            int(round(max_class_cnt * r_esi3)),
            int(round(max_class_cnt * r_esi4)),
            int(round(max_class_cnt * r_esi5))
        ]
        class_indices = [idx_2, idx_3, idx_4, idx_5]
        
        np.random.seed(42 + fold_idx)
        resampled_indices = []
        for idx_c, target_n in zip(class_indices, target_cnts):
            cur_n = len(idx_c)
            if cur_n == 0 or target_n <= 0: continue
            if cur_n < target_n:
                res_idx = np.random.choice(idx_c, size=target_n, replace=True)
            elif cur_n > target_n:
                res_idx = np.random.choice(idx_c, size=target_n, replace=False)
            else:
                res_idx = idx_c
            resampled_indices.append(res_idx)
            
        l2_resampled_idx = np.concatenate(resampled_indices)
        X_tr_l2 = X_tr_l2_raw[l2_resampled_idx]
        y_tr_l2_4cls = y_l2_raw[l2_resampled_idx].astype(int) - 2
        
        lgb_l2 = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, max_depth=6, random_state=42, verbose=-1)
        lgb_l2.fit(X_tr_l2, y_tr_l2_4cls)
        
        # 3. Soft Joint Probability Prediction on Validation Fold
        p_l1_esi1_val = lgb_l1.predict_proba(X_vl)[:, 1]
        p_l2_mat_val  = lgb_l2.predict_proba(X_vl)
        p_non1_val    = 1.0 - p_l1_esi1_val
        
        probs_val = np.zeros((len(val_idx), 5))
        probs_val[:, 0] = p_l1_esi1_val                            # ESI 1
        probs_val[:, 1] = p_non1_val * p_l2_mat_val[:, 0]           # ESI 2
        probs_val[:, 2] = p_non1_val * p_l2_mat_val[:, 1]           # ESI 3
        probs_val[:, 3] = p_non1_val * p_l2_mat_val[:, 2]           # ESI 4
        probs_val[:, 4] = p_non1_val * p_l2_mat_val[:, 3]           # ESI 5
        
        oof_probs[val_idx, :] = probs_val
        oof_preds[val_idx] = np.argmax(probs_val, axis=1) + 1
        
    oof_bal_acc = compute_macro_balanced_accuracy(y_int, oof_preds)
    return oof_bal_acc
print("Starting Optuna 5-Fold CV Layer 1 Class Weight & Layer 2 Resampling Search (20 Trials)...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=300)
print("\n============================================================")
print("   OPTUNA 5-FOLD CV CALIBRATION RESULTS")
print("============================================================")
print(f"  Best 5-Fold OOF Macro Balanced Accuracy : {study.best_value:.4f}")
print("  Winning Parameters:")
for k, v in study.best_params.items():
    print(f"    - {k:25s}: {v}")
print("============================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 3: Write Optimal Weight & Ratios directly to config/triage_conf.json
# ---------------------------------------------------------
conf_file = "../config/triage_conf.json"
if not os.path.exists(conf_file): conf_file = "config/triage_conf.json"
with open(conf_file, "r") as f:
    conf_data = json.load(f)
best_params = study.best_params
conf_data["resampling"] = {
    "layer1_weight_multiplier": float(best_params["l1_weight_multiplier"]),
    "layer2_class_ratios": {
        "esi2": float(best_params["r_esi2"]),
        "esi3": float(best_params["r_esi3"]),
        "esi4": float(best_params["r_esi4"]),
        "esi5": float(best_params["r_esi5"])
    }
}
with open(conf_file, "w") as f:
    json.dump(conf_data, f, indent=2)
print(f"SUCCESS: Optimal weight multiplier and resampling ratios saved to: {os.path.abspath(conf_file)}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Export Detailed Calibration Report & Line Plot
# ---------------------------------------------------------
import matplotlib.pyplot as plt
reports_dir = "../reports"
if not os.path.exists(reports_dir): reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
plots_dir = "../plots"
if not os.path.exists(plots_dir): plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)
trials_df = study.trials_dataframe()
trials_df = trials_df.rename(columns={
    'number': 'Trial_ID',
    'value': 'OOF_Macro_Balanced_Accuracy',
    'params_l1_weight_multiplier': 'l1_weight_multiplier',
    'params_r_esi2': 'r_esi2',
    'params_r_esi3': 'r_esi3',
    'params_r_esi4': 'r_esi4',
    'params_r_esi5': 'r_esi5'
})
trials_df['Best_So_Far'] = trials_df['OOF_Macro_Balanced_Accuracy'].cummax()
report_cols = ['Trial_ID', 'l1_weight_multiplier', 'r_esi2', 'r_esi3', 'r_esi4', 'r_esi5', 'OOF_Macro_Balanced_Accuracy', 'Best_So_Far']
res_report = trials_df[report_cols]
report_csv_path = os.path.join(reports_dir, "resampling_calibration_report.csv")
res_report.to_csv(report_csv_path, index=False)
print(f"Detailed Calibration Report written to: {report_csv_path}")
plt.figure(figsize=(10, 5.5), dpi=300)
plt.plot(trials_df['Trial_ID'], trials_df['OOF_Macro_Balanced_Accuracy'], marker='o', color='#1f77b4', linestyle='--', alpha=0.7, label='Trial OOF Macro BalAcc')
plt.plot(trials_df['Trial_ID'], trials_df['Best_So_Far'], color='#d62728', linewidth=2.5, label='Cumulative Best Score')
plt.title('5-Fold CV Layer 1 Class Weight & Layer 2 Resampling Calibration History', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Trial Index', fontsize=12)
plt.ylabel('OOF Macro Balanced Accuracy', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plot_path = os.path.join(plots_dir, "resampling_calibration_history.png")
plt.savefig(plot_path)
plt.close()
print(f"Calibration History Line Graph saved to: {plot_path}")